=============================================================================
UC2 — Returns Fraud Investigator
=============================================================================
Purpose  : Score every customer's return behaviour against 7 fraud rules,
           flag high-risk customers, and generate an AI investigation brief
           for each flagged customer using the Databricks LLM.

Reads    : globalmart.gold.fact_returns
           globalmart.gold.dim_customers
           globalmart.gold.fact_orders
           globalmart.gold.fraud_rule_config   (created in Cell 2)

Writes   : globalmart.gold.flagged_return_customers
           globalmart.gold.pipeline_run_log

Run order: Run cells top to bottom. Cell 2 seeds the rule config table
           on first run and is safe to re-run (it overwrites the table).
=============================================================================


In [0]:
# Run this cell first. The %pip install updates the cluster environment.
# dbutils.library.restartPython() resets the Python interpreter so the
%pip install openai
dbutils.library.restartPython()

All 7 fraud rules and their thresholds live in a Delta table, not in code.
This means a business analyst can tune thresholds (e.g. raise the flag
threshold during peak returns season) by running a simple SQL UPDATE on
globalmart.gold.fraud_rule_config — without touching this notebook.

Rule thresholds are derived from statistical analysis of the real dataset:
  800 returns  |  5006 orders  |  374 customers

| Rule Name | Threshold | Weight | What it measures |
|-----------|-----------|--------|------------------|
| **high_return_volume** | 5 | 25 pts | Customer has 5+ total returns.<br>Real data: 32 customers qualify, top at 8. |
| **high_refund_value** | $1,000 | 20 pts | Customer's total refunds >= $1,000.<br>Real data: 43 customers. Dataset mean = $584. |
| **high_order_return_rate** | 30% | 20 pts | >= 30% of the customer's orders were returned.<br>Calculated as total_returns / total_orders.<br>Always between 0–100%. Max in dataset = 50%.<br>No customer has more returns than orders. |
| **refund_inflation** | $500 | 25 pts | Customer received $500+ MORE in refunds than<br>the returned products were actually worth.<br>WHY this exists: original_sales = one product's<br>line-item value; refund_amount = full order<br>refund. Gap = real financial loss to GlobalMart.<br>Real data: 40 customers. Top = $1,743 excess. |
| **return_before_delivery** | 1 | 15 pts | At least 1 return was filed before the order<br>was delivered. Physically impossible.<br>Real data: 221 returns, avg gap = 563 days.<br>Could be legacy system date format issue too. |
| **duplicate_order_reason** | 1 | 10 pts | Customer used "Duplicate Order" as return<br>reason. Specific system exploit — claiming<br>the order was accidental to trigger a refund.<br>Real data: 31 returns, 28 customers. |
| **same_day_ship_return** | 1 | 10 pts | Customer returned an item from a Same Day<br>order. Return rate = 16.9% vs 16.0% average.<br>Weak standalone signal (+0.9pp), included as<br>a tiebreaker. Never flags a customer alone. |
| **flag_threshold** | 40 pts | — | Customers scoring >= 40 are flagged.<br>Requires triggering at least 2 rules.<br>Max possible score = 125 (all 7 rules). |

In [0]:


from pyspark.sql.types import StructType, StructField, StringType, FloatType 
import pyspark.sql.functions as F


thresholds = (
    spark.table("globalmart.gold.fraud_rule_config")
    .filter(F.col("is_active") == "True")
    .select("rule_name", "threshold", "weight")
    .toPandas()
    .set_index("rule_name")[["threshold", "weight"]]
    .to_dict(orient="index")
)

print("\nRules loaded for this run:")
print(f"  {'Rule':<30}  {'Threshold':>10}  {'Weight':>8}")
print(f"  {'-'*30}  {'-'*10}  {'-'*8}")
for rule, vals in thresholds.items():
    if rule != "flag_threshold":
        print(f"  {rule:<30}  {vals['threshold']:>10}  {vals['weight']:>8}")

flag_threshold = int(thresholds["flag_threshold"]["threshold"])
print(f"\n  Flag threshold: {flag_threshold} pts  (customers scoring >= {flag_threshold} are flagged)")

display(spark.table("globalmart.gold.fraud_rule_config"))

We use the OpenAI SDK pointed at the Databricks AI Gateway.
The API token is fetched automatically from the current Databricks session —
no hardcoded credentials anywhere.

Model: databricks-gpt-oss-20b
  - Free on Databricks Free Edition
  - Returns a structured list response: [reasoning block, text block]
  - We must extract only the "text" block (see extract_text() below)

In [0]:
import time
import json
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, FloatType
)
from openai import OpenAI

# Connect to the Databricks AI Gateway
client = OpenAI(
    api_key=dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get(),
    base_url="https://4092199907892374.ai-gateway.cloud.databricks.com/mlflow/v1/"
)

MODEL_NAME    = "databricks-gpt-oss-20b"
NOTEBOOK_NAME = "UC2_Returns_Fraud_Investigator"

print(f"LLM ready  : {MODEL_NAME}")
print(f"Run started: {datetime.now().isoformat()}")

In [0]:
# Three functions used by every LLM call in this notebook.

def extract_text(response) -> str:
    """
    Parse the model response and return only the answer text.

    Why this is needed:
    databricks-gpt-oss-20b returns a Python list, not a plain string:
      [{"type": "reasoning", "summary": [...]},   <- internal chain-of-thought
       {"type": "text",      "text": "answer"}]   <- the actual answer we want

    Without this function, printing response.content would include the
    model's raw reasoning chain alongside the answer, making the output unreadable.
    """
    content = response.choices[0].message.content

    # Structured list response — extract only the text block
    if isinstance(content, list):
        text_parts = [
            block["text"]
            for block in content
            if isinstance(block, dict) and block.get("type") == "text"
        ]
        return " ".join(text_parts).strip()

    # Plain string response (some gateway versions return this directly)
    return content.strip()


def call_llm(prompt: str, system_msg: str, retries: int = 3, wait_secs: int = 2) -> str:
    """
    Call the LLM with automatic retry on failure.

    Uses exponential backoff: wait 2s, then 4s, then 6s between attempts.
    On complete failure, returns an error string starting with LLM_UNAVAILABLE
    so the calling loop can continue processing other customers rather than
    crashing the entire notebook.
    """
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": system_msg},
                    {"role": "user",   "content": prompt}
                ],
                max_tokens=2000,
                temperature=0.3   # low temperature = consistent, grounded output
            )
            return extract_text(response)

        except Exception as e:
            if attempt < retries - 1:
                time.sleep(wait_secs * (attempt + 1))   # 2s, 4s, 6s
            else:
                return f"LLM_UNAVAILABLE: {str(e)}"     # safe fallback


def validate_output(text: str, min_words: int = 30) -> dict:
    """
    Check LLM output quality before writing to Delta.

    Catches three failure modes:
      too_short       — response was truncated (< 30 words)
      llm_call_failed — LLM was unreachable (starts with LLM_UNAVAILABLE)
      refusal_detected — model refused the request instead of answering

    Returns a dict with the text, a PASS/REVIEW flag, and a list of issues found.
    Only PASS outputs are written cleanly. REVIEW outputs are still written
    but flagged so operations can audit them.
    """
    refusal_phrases = ["i cannot", "as an ai", "i don't have", "i'm unable"]
    issues = []

    if len(text.split()) < min_words:
        issues.append("too_short")
    if text.startswith("LLM_UNAVAILABLE"):
        issues.append("llm_call_failed")
    if any(phrase in text.lower() for phrase in refusal_phrases):
        issues.append("refusal_detected")

    return {
        "text":         text,
        "llm_check": "PASS" if not issues else "REVIEW",
        "issues":       ", ".join(issues) if issues else "none"
    }


print("LLM helper functions defined.")

In [0]:
# Load the three Gold layer tables this notebook reads from.
# All data is already clean and harmonised — this notebook only reads Gold,
# it does not touch Bronze or Silver.

df_returns   = spark.table("globalmart.gold.fact_returns")
df_customers = spark.table("globalmart.gold.dim_customers")
df_orders    = spark.table("globalmart.gold.fact_orders")

print("Tables loaded:")
print(f"  fact_returns   : {df_returns.count():,} rows")
print(f"  dim_customers  : {df_customers.count():,} rows")
print(f"  fact_orders    : {df_orders.count():,} rows")

display(df_returns.limit(5))

In [0]:
# On the first run there is no output table yet, so we score every customer.
# On every run after that, we only re-score customers who have had new return
# activity since the last run — skipping customers whose profiles have not changed.
#
# This keeps the notebook fast and avoids regenerating identical LLM briefs
# for customers who were already scored and have not done anything new.

try:
    # Check when this notebook last ran successfully
    last_run_ts = spark.sql(
        "SELECT MAX(scored_at) FROM globalmart.gold.flagged_return_customers"
    ).collect()[0][0]
    print(f"Last run: {last_run_ts}")
    print("Incremental mode: only processing customers with new returns since last run.")

except Exception:
    # Table does not exist yet — this is the first run
    last_run_ts = None
    print("First run detected. Scoring all customers.")

# Filter returns to only include new activity if a previous run exists
if last_run_ts is not None and "return_date" in [f.name for f in df_returns.schema]:
    # Find customer IDs who have any new returns since last run
    changed_customers = (
        df_returns
        .filter(F.col("return_date") > F.lit(str(last_run_ts)))
        .select("customer_id")
        .distinct()
    )
    # Pull ALL returns for those customers (not just the new ones)
    # so their full return profile is scored correctly
    df_returns_to_score = df_returns.join(changed_customers, on="customer_id", how="inner")
    print(f"Customers to re-score: {changed_customers.count():,}")

else:
    # First run — score everyone
    df_returns_to_score = df_returns
    print(f"Scoring all {df_returns.select('customer_id').distinct().count():,} customers.")

In [0]:
# We score each CUSTOMER, not each individual return row.
# This cell computes 7 per-customer signals — one for each fraud rule.
# Each signal is a separate aggregation that gets joined into one profile table.
#
# Schema note on refund inflation (Rule 4):
#   original_sales  = the sale value of the SINGLE product being returned
#                     (one line item from the transactions table)
#   refund_amount   = the FULL ORDER refund issued by GlobalMart
#   line_inflation  = refund_amount - original_sales (the gap = money lost)
#
# Example from real data:
#   Product sold for $1.63, full order refunded at $480.36 → $478.73 inflation
#   This is real financial loss, not a data error.

# ── Signal 1 & 2: Basic return counts and total refund value ─────────────────
df_base = (
    df_returns_to_score
    .groupBy("customer_id")
    .agg(
        F.count("*")                        .alias("total_returns"),
        F.sum("refund_amount")              .alias("total_refund_value"),
        F.countDistinct("return_reason")    .alias("distinct_reasons"),
        F.collect_set("return_reason")      .alias("return_reasons_list")
    )
)

# ── Signal 4: Refund inflation ────────────────────────────────────────────────
# Only calculated on rows where both amounts are present (neither is NULL)
df_inflation = (
    df_returns_to_score
    .filter(F.col("refund_amount").isNotNull() & F.col("original_sales").isNotNull())
    .withColumn("line_inflation", F.col("refund_amount") - F.col("original_sales"))
    .groupBy("customer_id")
    .agg(
        F.sum("line_inflation")                             .alias("total_refund_inflation"),
        F.max("line_inflation")                             .alias("max_single_inflation"),
        F.count(F.when(F.col("line_inflation") > 0, 1))    .alias("inflated_refund_count")
    )
)

# ── Signal 6: Duplicate Order return reason ───────────────────────────────────
df_duplicate_reason = (
    df_returns_to_score
    .filter(F.col("return_reason") == "Duplicate Order")
    .groupBy("customer_id")
    .agg(F.count("*").alias("duplicate_order_count"))
)

# ── Signals 5 & 7: Timing signals (return before delivery, same-day shipping) ─
# The order timestamp has a timezone suffix (e.g. "2017-09-29T18:38:00.000Z").
# We strip everything from "T" onwards to get a plain date string before parsing.
df_orders_with_dates = (
    df_orders.select(
        "order_id",
        "ship_mode",
        F.to_date(
            F.regexp_replace("order_delivered_customer_timestamp", "T.*", "")
        ).alias("delivered_date")
    )
)

df_timing = (
    df_returns_to_score
    .select(
        "customer_id",
        "order_id",
        F.to_date("return_date").alias("return_date_parsed")
    )
    .join(df_orders_with_dates, on="order_id", how="left")

    # Flag: return filed before the order was even delivered
    .withColumn("is_before_delivery",
        F.when(
            F.col("return_date_parsed").isNotNull() &
            F.col("delivered_date").isNotNull() &
            (F.col("return_date_parsed") < F.col("delivered_date")),
            1
        ).otherwise(0)
    )

    # Flag: this return came from a Same Day shipping order
    .withColumn("is_same_day",
        F.when(F.col("ship_mode") == "Same Day", 1).otherwise(0))
)

df_timing_signals = (
    df_timing
    .groupBy("customer_id")
    .agg(
        F.sum("is_before_delivery") .alias("return_before_delivery_count"),
        F.sum("is_same_day")        .alias("same_day_ship_return_count")
    )
)

# ── Signal 3: True order return rate ─────────────────────────────────────────
# = total_returns / total_orders per customer
# This ratio is ALWAYS between 0.0 and 1.0. Max in this dataset = 0.50.
# No customer has more returns than orders (verified in data analysis).
df_order_counts = (
    df_orders
    .groupBy("customer_id")
    .agg(F.count("*").alias("total_orders"))
)

# ── Combine all signals into one profile per customer ─────────────────────────
df_profiles = (
    df_base
    .join(df_inflation,        on="customer_id", how="left")
    .join(df_duplicate_reason, on="customer_id", how="left")
    .join(df_timing_signals,   on="customer_id", how="left")
    .join(df_order_counts,     on="customer_id", how="left")
    .join(
        df_customers.select("customer_id", "customer_name", "segment", "region"),
        on="customer_id",
        how="left"
    )
    # Replace NULLs with zeros for customers who have no activity in a signal
    .fillna({
        "total_orders":                  0,
        "total_refund_value":            0.0,
        "total_refund_inflation":        0.0,
        "max_single_inflation":          0.0,
        "inflated_refund_count":         0,
        "duplicate_order_count":         0,
        "return_before_delivery_count":  0,
        "same_day_ship_return_count":    0,
    })
    # Derived column: order return rate (always 0.0–1.0)
    .withColumn("order_return_rate",
        F.when(
            F.col("total_orders") > 0,
            F.col("total_returns") / F.col("total_orders")
        ).otherwise(F.lit(0.0))
    )
)

print(f"Profiles built for {df_profiles.count():,} customers.")
display(
    df_profiles
    .select(
        "customer_id", "customer_name", "total_returns", "total_orders",
        "order_return_rate", "total_refund_value",
        "total_refund_inflation", "return_before_delivery_count",
        "duplicate_order_count", "same_day_ship_return_count"
    )
    .orderBy(F.col("total_returns").desc())
    .limit(15)
)

In [0]:
# Each rule adds a score column: the rule's weight if triggered, 0 if not.
# A customer's total_score is the sum of all triggered rule weights.
# Maximum possible score = 125 (all 7 rules triggered simultaneously).
#
# Thresholds and weights come from the fraud_rule_config table loaded in Cell 2,
# not from hardcoded values here. If a business analyst changes a threshold in
# that table, the new value takes effect on the next run of this notebook.

# Pull thresholds (t_) and weights (w_) from the config dict
t_vol    = float(thresholds["high_return_volume"]["threshold"])
t_refund = float(thresholds["high_refund_value"]["threshold"])
t_rate   = float(thresholds["high_order_return_rate"]["threshold"])
t_infl   = float(thresholds["refund_inflation"]["threshold"])
t_before = float(thresholds["return_before_delivery"]["threshold"])
t_dup    = float(thresholds["duplicate_order_reason"]["threshold"])
t_same   = float(thresholds["same_day_ship_return"]["threshold"])
THRESHOLD = int(thresholds["flag_threshold"]["threshold"])

w_vol    = float(thresholds["high_return_volume"]["weight"])
w_refund = float(thresholds["high_refund_value"]["weight"])
w_rate   = float(thresholds["high_order_return_rate"]["weight"])
w_infl   = float(thresholds["refund_inflation"]["weight"])
w_before = float(thresholds["return_before_delivery"]["weight"])
w_dup    = float(thresholds["duplicate_order_reason"]["weight"])
w_same   = float(thresholds["same_day_ship_return"]["weight"])

df_scored = (
    df_profiles

    # Rule 1 — High return volume
    # 32 customers in dataset have 5+ returns. Top customers: 8 returns each.
    .withColumn("score_r1_volume",
        F.when(F.col("total_returns") >= t_vol, w_vol).otherwise(0))

    # Rule 2 — High total refund value
    # 43 customers exceed $1,000 total. Dataset mean = $584.
    .withColumn("score_r2_refund_value",
        F.when(F.col("total_refund_value") >= t_refund, w_refund).otherwise(0))

    # Rule 3 — High order return rate
    # order_return_rate = total_returns / total_orders (always 0.0 to 1.0)
    # 30% means 3 in every 10 of this customer's orders ended in a return.
    .withColumn("score_r3_return_rate",
        F.when(F.col("order_return_rate") >= t_rate, w_rate).otherwise(0))

    # Rule 4 — Refund inflation
    # total_refund_inflation = SUM(refund_amount - original_sales)
    # Positive inflation = customer received more money back than products cost.
    # 40 customers exceed $500. Top offender: $1,743 excess received.
    .withColumn("score_r4_inflation",
        F.when(F.col("total_refund_inflation") >= t_infl, w_infl).otherwise(0))

    # Rule 5 — Return filed before delivery date
    # 221 returns across 160 customers have return_date < delivered_date.
    # Average gap: 563 days before delivery. Cannot be explained by clock skew.
    .withColumn("score_r5_before_delivery",
        F.when(F.col("return_before_delivery_count") >= t_before, w_before).otherwise(0))

    # Rule 6 — Duplicate Order return reason
    # Customers claiming an order was a duplicate to get a refund on a real purchase.
    # Only 31 such returns in the dataset — low frequency makes it more suspicious.
    .withColumn("score_r6_duplicate_reason",
        F.when(F.col("duplicate_order_count") >= t_dup, w_dup).otherwise(0))

    # Rule 7 — Return from a Same Day shipping order
    # Return rate for Same Day = 16.9% vs 16.0% overall (+0.9pp).
    # Weak standalone signal — only adds value in combination with other rules.
    # Lowest weight (10 pts) — a customer triggering only this rule scores 10,
    # which is below the 40-point flag threshold, so they are never flagged alone.
    .withColumn("score_r7_same_day",
        F.when(F.col("same_day_ship_return_count") >= t_same, w_same).otherwise(0))

    # Total risk score (max = 125)
    .withColumn("total_score",
        F.col("score_r1_volume")          +
        F.col("score_r2_refund_value")    +
        F.col("score_r3_return_rate")     +
        F.col("score_r4_inflation")       +
        F.col("score_r5_before_delivery") +
        F.col("score_r6_duplicate_reason")+
        F.col("score_r7_same_day"))
)

print("Scoring complete. Top 15 customers by risk score:")
display(
    df_scored
    .select(
        "customer_id", "customer_name", "total_returns", "total_orders",
        "order_return_rate", "total_refund_inflation",
        "return_before_delivery_count", "total_score"
    )
    .orderBy(F.col("total_score").desc())
    .limit(15)
)

In [0]:
# Filter to customers whose total_score meets or exceeds the flag threshold (40).
# These are the customers who need an investigation brief.

df_flagged    = df_scored.filter(F.col("total_score") >= THRESHOLD)
flagged_count = df_flagged.count()

print(f"Flagged {flagged_count:,} customers for investigation  (score >= {THRESHOLD})")
print(f"Total customers scored: {df_scored.count():,}")
display(df_flagged.orderBy(F.col("total_score").desc()))

In [0]:
# For each flagged customer, we call the LLM to produce a 3-4 sentence
# investigation brief for the returns operations manager.
#
# Design principle: rules DETECT, LLM EXPLAINS.
# The scoring above identifies suspicious customers using deterministic rules.
# The LLM only explains WHY a customer was flagged in plain business English —
# it does not make any detection decisions itself.
#
# The prompt includes:
#   - The customer's actual data values (not placeholders)
#   - Which specific rules were violated and by how much
#   - Background on what refund inflation means in this schema
#   - A clear instruction to write for an operations manager (non-technical)

def safe(row, field, default="N/A"):
    """
    Safely read a field from a Spark Row object.
    Spark Rows do not support .get() like Python dicts,
    so we use try/except to avoid KeyErrors.
    """
    try:
        val = row[field]
        return val if val is not None else default
    except Exception:
        return default


def build_prompt(row) -> str:
    """
    Build the LLM prompt for one flagged customer.
    All values are taken from the actual scored row — no generic text.
    """

    # Build the rules violated section with actual figures from the data
    violated_rules = []

    if row["score_r1_volume"] > 0:
        violated_rules.append(
            f"  - High return volume : {int(safe(row,'total_returns',0))} total returns"
            f" (threshold = {int(t_vol)}) | {int(w_vol)} pts")

    if row["score_r2_refund_value"] > 0:
        violated_rules.append(
            f"  - High refund value  : ${round(safe(row,'total_refund_value',0), 2)} total"
            f" (threshold = ${int(t_refund)}) | {int(w_refund)} pts")

    if row["score_r3_return_rate"] > 0:
        violated_rules.append(
            f"  - High return rate   : {round(safe(row,'order_return_rate',0)*100, 1)}% of orders returned"
            f" (threshold = {int(t_rate*100)}%) | {int(w_rate)} pts")

    if row["score_r4_inflation"] > 0:
        violated_rules.append(
            f"  - Refund inflation   : ${round(safe(row,'total_refund_inflation',0), 2)} received above product values"
            f" (threshold = ${int(t_infl)}) | {int(w_infl)} pts")

    if row["score_r5_before_delivery"] > 0:
        violated_rules.append(
            f"  - Before delivery    : {int(safe(row,'return_before_delivery_count',0))} returns filed before delivery"
            f" | {int(w_before)} pts")

    if row["score_r6_duplicate_reason"] > 0:
        violated_rules.append(
            f"  - Duplicate Order    : {int(safe(row,'duplicate_order_count',0))} returns claiming Duplicate Order reason"
            f" | {int(w_dup)} pts")

    if row["score_r7_same_day"] > 0:
        violated_rules.append(
            f"  - Same Day returns   : {int(safe(row,'same_day_ship_return_count',0))} returns on Same Day orders"
            f" | {int(w_same)} pts")

    reasons_text = ", ".join(safe(row, "return_reasons_list", []) or []) or "Not recorded"

    return f"""You are a fraud analyst at GlobalMart, a national retail chain.

A customer has been flagged by the automated anomaly scoring system with a risk score
of {row['total_score']} out of 125. Your job is to explain these patterns clearly
for the returns operations manager who will decide whether to investigate further.

SCHEMA NOTE:
  original_sales  = the value of the single product returned (one line item)
  refund_amount   = the full order refund issued (may cover multiple products)
  Refund inflation = the difference — money GlobalMart refunded above the product's value

CUSTOMER:
  ID              : {safe(row, 'customer_id')}
  Name            : {safe(row, 'customer_name')}
  Segment         : {safe(row, 'segment')}
  Region          : {safe(row, 'region')}

THEIR RETURN BEHAVIOUR:
  Total returns           : {int(safe(row, 'total_returns', 0))}
  Total orders placed     : {int(safe(row, 'total_orders', 0))}
  Order return rate       : {round(safe(row, 'order_return_rate', 0)*100, 1)}%
  Total refunded          : ${round(safe(row, 'total_refund_value', 0), 2)}
  Total refund inflation  : ${round(safe(row, 'total_refund_inflation', 0), 2)}
  Largest single inflation: ${round(safe(row, 'max_single_inflation', 0), 2)}
  Returns before delivery : {int(safe(row, 'return_before_delivery_count', 0))}
  Duplicate Order claims  : {int(safe(row, 'duplicate_order_count', 0))}
  Same Day returns        : {int(safe(row, 'same_day_ship_return_count', 0))}
  Return reasons used     : {reasons_text}

RULES VIOLATED (risk score = {row['total_score']} / 125):
{chr(10).join(violated_rules)}

Write a 3–4 sentence investigation brief for the returns operations manager. Cover:
1. What specific numbers make this case suspicious — cite the actual figures above
2. What innocent explanations might exist for these patterns
3. What the returns team should check first — be specific and actionable

Write in plain English. No bullet points. No markdown. No section headers."""


# System message tells the LLM its role for this task
SYSTEM_MSG = (
    "You are a fraud analyst writing clear, data-grounded investigation briefs "
    "for a returns operations manager at GlobalMart retail. "
    "Write in plain connected English paragraphs. No bullet points, no markdown."
)

# Loop over each flagged customer, call the LLM, collect results
fraud_records  = []
llm_calls_made = 0
needs_review   = 0

for row in df_flagged.orderBy(F.col("total_score").desc()).collect():

    # Build and send the prompt
    prompt    = build_prompt(row)
    raw_text  = call_llm(prompt, SYSTEM_MSG)
    result    = validate_output(raw_text)
    llm_calls_made += 1

    if result["llm_check"] == "REVIEW":
        needs_review += 1

    # Build the pipe-delimited list of rule names that fired
    rules_fired = "|".join([
        name for name, score in [
            ("high_return_volume",       row["score_r1_volume"]),
            ("high_refund_value",        row["score_r2_refund_value"]),
            ("high_order_return_rate",   row["score_r3_return_rate"]),
            ("refund_inflation",         row["score_r4_inflation"]),
            ("return_before_delivery",   row["score_r5_before_delivery"]),
            ("duplicate_order_reason",   row["score_r6_duplicate_reason"]),
            ("same_day_ship_return",     row["score_r7_same_day"]),
        ] if score > 0
    ])

    # Collect the record for Delta write
    fraud_records.append({
        "customer_id":                   safe(row, "customer_id"),
        "customer_name":                 safe(row, "customer_name"),
        "segment":                       safe(row, "segment"),
        "region":                        safe(row, "region"),
        "total_returns":                 int(safe(row, "total_returns", 0)),
        "total_orders":                  int(safe(row, "total_orders",  0)),
        "order_return_rate":             float(round(safe(row, "order_return_rate", 0), 4)),
        "total_refund_value":            float(round(safe(row, "total_refund_value", 0), 2)),
        "total_refund_inflation":        float(round(safe(row, "total_refund_inflation", 0), 2)),
        "max_single_inflation":          float(round(safe(row, "max_single_inflation", 0), 2)),
        "inflated_refund_count":         int(safe(row, "inflated_refund_count", 0)),
        "return_before_delivery_count":  int(safe(row, "return_before_delivery_count", 0)),
        "duplicate_order_count":         int(safe(row, "duplicate_order_count", 0)),
        "same_day_ship_return_count":    int(safe(row, "same_day_ship_return_count", 0)),
        "anomaly_score":                 int(row["total_score"]),
        "rules_violated":                rules_fired,
        "investigation_brief":           result["text"],
        "llm_check":                  result["llm_check"],
        "llm_check_detail":                result["issues"],
        "scored_at":                     datetime.now().isoformat()
    })

    # Print a readable summary for each flagged customer
    print("=" * 65)
    print(f"Customer : {safe(row,'customer_id')}  |  {safe(row,'customer_name')}")
    print(f"Segment  : {safe(row,'segment')}  |  Region: {safe(row,'region')}")
    print(f"Score    : {row['total_score']} / 125")
    print(f"Rules    : {rules_fired}")
    print(f"Quality  : {result['llm_check']}")
    print()
    print("BRIEF:")
    print(result["text"])
    print("=" * 65)
    print()

print(f"Done.  Flagged: {len(fraud_records)}  |  LLM calls: {llm_calls_made}  |  Needs review: {needs_review}")

In [0]:
# Write all flagged customer records to the output Delta table.
# Mode = "append" so each run adds new records without overwriting history.
# mergeSchema = True handles any new columns added in future without breaking.

output_schema = StructType([
    StructField("customer_id",                  StringType(),  True),
    StructField("customer_name",                StringType(),  True),
    StructField("segment",                      StringType(),  True),
    StructField("region",                       StringType(),  True),
    StructField("total_returns",                IntegerType(), True),
    StructField("total_orders",                 IntegerType(), True),
    StructField("order_return_rate",            FloatType(),   True),
    StructField("total_refund_value",           FloatType(),   True),
    StructField("total_refund_inflation",       FloatType(),   True),
    StructField("max_single_inflation",         FloatType(),   True),
    StructField("inflated_refund_count",        IntegerType(), True),
    StructField("return_before_delivery_count", IntegerType(), True),
    StructField("duplicate_order_count",        IntegerType(), True),
    StructField("same_day_ship_return_count",   IntegerType(), True),
    StructField("anomaly_score",                IntegerType(), True),
    StructField("rules_violated",               StringType(),  True),
    StructField("investigation_brief",          StringType(),  True),
    StructField("llm_check",                 StringType(),  True),
    StructField("llm_check_detail",               StringType(),  True),
    StructField("scored_at",                    StringType(),  True),
])

df_output = spark.createDataFrame(fraud_records, schema=output_schema)

(
    df_output
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable("globalmart.gold.flagged_return_customers")
)

print("Saved to globalmart.gold.flagged_return_customers")
display(
    spark.table("globalmart.gold.flagged_return_customers")
    .orderBy(F.col("anomaly_score").desc())
)

In [0]:
# Append one row to the operations run log so anyone can query:
#   - When did this notebook last run?
#   - How many customers were scored and flagged?
#   - Did any LLM outputs need manual review?
#   - Was incremental (watermark) mode active?

run_log_entry = [{
    "notebook":          NOTEBOOK_NAME,
    "run_timestamp":     datetime.now().isoformat(),
    "records_processed": df_profiles.count(),
    "records_flagged":   flagged_count,
    "llm_calls_made":    llm_calls_made,
    "outputs_flagged":   needs_review,
    "status":            "SUCCESS",
    "notes":             (
        f"flag_threshold={THRESHOLD} | "
        f"rules=7 | "
        f"incremental={last_run_ts is not None}"
    )
}]

(
    spark.createDataFrame(run_log_entry)
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable("globalmart.gold.pipeline_run_log")
)

print(f"Run logged.  Status: SUCCESS")
print(f"Processed: {df_profiles.count():,} customers  |  Flagged: {flagged_count}  |  Review needed: {needs_review}")